# 03 — Feature Engineering

**Objective**: Read all silver layer tables, engineer aggregated features from each auxiliary table, merge everything at the `SK_ID_CURR` level, and save the gold layer feature datasets.

## Feature Engineering Strategy

Each auxiliary table (bureau, previous applications, POS cash, credit card, installments) contains multiple rows per client. We aggregate them to the `SK_ID_CURR` level using statistical summaries (mean, min, max, count, ratios) to create a flat feature matrix.

| Source Table | Key Features |
|---|---|
| `bureau` | Credit history stats, active/closed counts, overdue proportions |
| `bureau_balance` | DPD status counts per client |
| `previous_application` | Application stats, approval/refusal counts, credit ratios |
| `pos_cash_balance` | POS loan DPD stats, completed contracts |
| `credit_card_balance` | Utilization ratios, drawing patterns, DPD stats |
| `installments_payments` | Payment differences, late payment rates |

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import gc

warnings.filterwarnings("ignore")

# Project paths
PROJECT_ROOT = Path("..").resolve()
SILVER_DIR = PROJECT_ROOT / "data" / "processed"   # Silver layer (input)
GOLD_DIR   = PROJECT_ROOT / "data" / "features"    # Gold layer (output)

GOLD_DIR.mkdir(parents=True, exist_ok=True)

print(f"Silver layer: {SILVER_DIR}")
print(f"Gold layer:   {GOLD_DIR}")

## 1. Load All Silver Tables

In [ ]:
# Load all silver layer tables
app_train = pd.read_parquet(SILVER_DIR / "application_train.parquet")
app_test  = pd.read_parquet(SILVER_DIR / "application_test.parquet")
bureau    = pd.read_parquet(SILVER_DIR / "bureau.parquet")
bb        = pd.read_parquet(SILVER_DIR / "bureau_balance.parquet")
prev      = pd.read_parquet(SILVER_DIR / "previous_application.parquet")
pos       = pd.read_parquet(SILVER_DIR / "pos_cash_balance.parquet")
cc        = pd.read_parquet(SILVER_DIR / "credit_card_balance.parquet")
inst      = pd.read_parquet(SILVER_DIR / "installments_payments.parquet")

print("Silver layer tables loaded:\n")
for name, df in [("application_train", app_train), ("application_test", app_test),
                 ("bureau", bureau), ("bureau_balance", bb),
                 ("previous_application", prev), ("pos_cash_balance", pos),
                 ("credit_card_balance", cc), ("installments_payments", inst)]:
    print(f"  {name:25s} {df.shape[0]:>12,} rows × {df.shape[1]:>3} cols")

## 2. Bureau Features

Aggregate bureau credit history to the `SK_ID_CURR` level. This captures the client's credit history from other financial institutions.

In [ ]:
def aggregate_bureau_features(bureau):
    """Aggregate bureau table features to SK_ID_CURR level."""
    
    # --- Numeric aggregations ---
    num_agg = {
        "DAYS_CREDIT":           ["count", "mean", "min", "max"],
        "AMT_CREDIT_SUM":        ["mean", "min", "max", "sum"],
        "AMT_CREDIT_SUM_DEBT":   ["mean", "min", "max", "sum"],
        "AMT_CREDIT_SUM_OVERDUE": ["mean", "max"],
        "DAYS_CREDIT_ENDDATE":   ["mean", "min", "max"],
        "AMT_CREDIT_MAX_OVERDUE": ["mean", "max"],
        "AMT_ANNUITY":           ["mean", "min", "max"],
        "CREDIT_DAY_OVERDUE":    ["mean", "max"],
        "DAYS_CREDIT_UPDATE":    ["mean", "min"],
        "CNT_CREDIT_PROLONG":    ["sum", "max"],
    }
    
    bureau_agg = bureau.groupby("SK_ID_CURR").agg(num_agg)
    bureau_agg.columns = [f"BUREAU_{col[0]}_{col[1].upper()}" for col in bureau_agg.columns]
    
    # Rename count column for clarity
    bureau_agg.rename(columns={"BUREAU_DAYS_CREDIT_COUNT": "BUREAU_CREDIT_COUNT"}, inplace=True)
    
    # --- Active vs Closed credit counts ---
    credit_active = bureau.groupby("SK_ID_CURR")["CREDIT_ACTIVE"].value_counts().unstack(fill_value=0)
    for col in credit_active.columns:
        safe_col = str(col).replace(" ", "_").upper()
        bureau_agg[f"BUREAU_ACTIVE_{safe_col}_COUNT"] = credit_active[col]
    
    # --- Proportion of overdue credits ---
    bureau_agg["BUREAU_OVERDUE_RATIO"] = (
        bureau_agg["BUREAU_AMT_CREDIT_SUM_OVERDUE_MAX"] > 0
    ).astype(np.float32)
    
    # Overdue proportion based on actual overdue amounts
    overdue_counts = bureau[bureau["CREDIT_DAY_OVERDUE"] > 0].groupby("SK_ID_CURR").size()
    total_counts = bureau.groupby("SK_ID_CURR").size()
    bureau_agg["BUREAU_OVERDUE_CREDIT_PROPORTION"] = (overdue_counts / total_counts).astype(np.float32)
    bureau_agg["BUREAU_OVERDUE_CREDIT_PROPORTION"].fillna(0, inplace=True)
    
    bureau_agg.reset_index(inplace=True)
    
    print(f"  Bureau features:         {bureau_agg.shape[1] - 1:>3} features for {bureau_agg.shape[0]:>7,} clients")
    return bureau_agg


def aggregate_bureau_balance_features(bb, bureau):
    """Aggregate bureau_balance DPD status counts to SK_ID_CURR level."""
    
    # Map STATUS to DPD indicator (0 = no DPD, C = closed, X = unknown, 1-5 = months DPD)
    # Statuses 1, 2, 3, 4, 5 indicate days past due
    bb_copy = bb.copy()
    bb_copy["STATUS"] = bb_copy["STATUS"].astype(str)
    bb_copy["BB_DPD_FLAG"] = bb_copy["STATUS"].isin(["1", "2", "3", "4", "5"]).astype(np.int8)
    
    # Aggregate per bureau record
    bb_agg = bb_copy.groupby("SK_ID_BUREAU").agg(
        BB_MONTHS_COUNT=("MONTHS_BALANCE", "count"),
        BB_DPD_MONTHS_COUNT=("BB_DPD_FLAG", "sum"),
    ).reset_index()
    
    bb_agg["BB_DPD_MONTHS_RATIO"] = (
        bb_agg["BB_DPD_MONTHS_COUNT"] / bb_agg["BB_MONTHS_COUNT"]
    ).astype(np.float32)
    
    # Join with bureau to get SK_ID_CURR
    bureau_key = bureau[["SK_ID_BUREAU", "SK_ID_CURR"]].drop_duplicates()
    bb_agg = bb_agg.merge(bureau_key, on="SK_ID_BUREAU", how="left")
    
    # Aggregate to client level
    bb_client = bb_agg.groupby("SK_ID_CURR").agg(
        BB_TOTAL_MONTHS_COUNT=("BB_MONTHS_COUNT", "sum"),
        BB_TOTAL_DPD_MONTHS=("BB_DPD_MONTHS_COUNT", "sum"),
        BB_MEAN_DPD_RATIO=("BB_DPD_MONTHS_RATIO", "mean"),
        BB_MAX_DPD_RATIO=("BB_DPD_MONTHS_RATIO", "max"),
    ).reset_index()
    
    bb_client["BB_OVERALL_DPD_RATIO"] = (
        bb_client["BB_TOTAL_DPD_MONTHS"] / bb_client["BB_TOTAL_MONTHS_COUNT"]
    ).astype(np.float32)
    
    print(f"  Bureau balance features: {bb_client.shape[1] - 1:>3} features for {bb_client.shape[0]:>7,} clients")
    return bb_client


print("Aggregating bureau features...\n")
bureau_features = aggregate_bureau_features(bureau)
bb_features = aggregate_bureau_balance_features(bb, bureau)

del bb
gc.collect()
print(f"\n  Memory freed after bureau_balance processing.")

## 3. Previous Application Features

Aggregate previous Home Credit applications to the `SK_ID_CURR` level.

In [ ]:
def aggregate_previous_application_features(prev):
    """Aggregate previous_application table to SK_ID_CURR level."""
    
    # --- Numeric aggregations ---
    num_agg = {
        "AMT_ANNUITY":      ["mean", "min", "max"],
        "AMT_APPLICATION":  ["mean", "min", "max"],
        "AMT_CREDIT":       ["mean", "min", "max", "sum"],
        "AMT_DOWN_PAYMENT":  ["mean", "min", "max"],
        "AMT_GOODS_PRICE":  ["mean", "min", "max"],
        "DAYS_DECISION":    ["mean", "min", "max"],
    }
    
    prev_agg = prev.groupby("SK_ID_CURR").agg(num_agg)
    prev_agg.columns = [f"PREV_{col[0]}_{col[1].upper()}" for col in prev_agg.columns]
    
    # --- Count of previous applications ---
    prev_agg["PREV_APPLICATION_COUNT"] = prev.groupby("SK_ID_CURR").size()
    
    # --- Counts by contract status ---
    status_counts = prev.groupby("SK_ID_CURR")["NAME_CONTRACT_STATUS"].value_counts().unstack(fill_value=0)
    for col in status_counts.columns:
        safe_col = str(col).replace(" ", "_").replace("'", "").upper()
        prev_agg[f"PREV_STATUS_{safe_col}_COUNT"] = status_counts[col]
    
    # --- Approval rate ---
    if "PREV_STATUS_APPROVED_COUNT" in prev_agg.columns:
        prev_agg["PREV_APPROVAL_RATE"] = (
            prev_agg["PREV_STATUS_APPROVED_COUNT"] / prev_agg["PREV_APPLICATION_COUNT"]
        ).astype(np.float32)
    
    # --- Application-to-credit ratio ---
    prev_agg["PREV_APP_CREDIT_RATIO_MEAN"] = (
        prev_agg["PREV_AMT_APPLICATION_MEAN"] / prev_agg["PREV_AMT_CREDIT_MEAN"].replace(0, np.nan)
    ).astype(np.float32)
    
    prev_agg.reset_index(inplace=True)
    
    print(f"  Previous app features:   {prev_agg.shape[1] - 1:>3} features for {prev_agg.shape[0]:>7,} clients")
    return prev_agg


print("Aggregating previous application features...\n")
prev_features = aggregate_previous_application_features(prev)

del prev
gc.collect()
print(f"\n  Memory freed after previous_application processing.")

## 4. POS Cash Balance Features

Aggregate POS/cash loan monthly snapshots to the `SK_ID_CURR` level.

In [ ]:
def aggregate_pos_cash_features(pos):
    """Aggregate pos_cash_balance table to SK_ID_CURR level."""
    
    pos_agg = pos.groupby("SK_ID_CURR").agg(
        POS_RECORD_COUNT=("SK_ID_PREV", "count"),
        POS_SK_DPD_MEAN=("SK_DPD", "mean"),
        POS_SK_DPD_MAX=("SK_DPD", "max"),
        POS_SK_DPD_DEF_MEAN=("SK_DPD_DEF", "mean"),
        POS_SK_DPD_DEF_MAX=("SK_DPD_DEF", "max"),
        POS_MONTHS_BALANCE_MEAN=("MONTHS_BALANCE", "mean"),
        POS_CNT_INSTALMENT_MEAN=("CNT_INSTALMENT", "mean"),
        POS_CNT_INSTALMENT_FUTURE_MEAN=("CNT_INSTALMENT_FUTURE", "mean"),
    ).reset_index()
    
    # Count of completed contracts
    pos_str = pos.copy()
    pos_str["NAME_CONTRACT_STATUS"] = pos_str["NAME_CONTRACT_STATUS"].astype(str)
    completed = pos_str[pos_str["NAME_CONTRACT_STATUS"] == "Completed"].groupby("SK_ID_CURR").size()
    pos_agg["POS_COMPLETED_COUNT"] = pos_agg["SK_ID_CURR"].map(completed).fillna(0).astype(np.int32)
    
    # Count of unique POS contracts
    pos_agg["POS_CONTRACT_COUNT"] = pos.groupby("SK_ID_CURR")["SK_ID_PREV"].nunique().values
    
    # Convert float columns
    float_cols = pos_agg.select_dtypes(include=["float64"]).columns
    pos_agg[float_cols] = pos_agg[float_cols].astype(np.float32)
    
    print(f"  POS cash features:       {pos_agg.shape[1] - 1:>3} features for {pos_agg.shape[0]:>7,} clients")
    return pos_agg


print("Aggregating POS cash balance features...\n")
pos_features = aggregate_pos_cash_features(pos)

del pos
gc.collect()
print(f"\n  Memory freed after pos_cash_balance processing.")

## 5. Credit Card Balance Features

Aggregate credit card monthly snapshots to the `SK_ID_CURR` level.

In [ ]:
def aggregate_credit_card_features(cc):
    """Aggregate credit_card_balance table to SK_ID_CURR level."""
    
    cc_agg = cc.groupby("SK_ID_CURR").agg(
        CC_RECORD_COUNT=("SK_ID_PREV", "count"),
        CC_AMT_BALANCE_MEAN=("AMT_BALANCE", "mean"),
        CC_AMT_BALANCE_MAX=("AMT_BALANCE", "max"),
        CC_AMT_CREDIT_LIMIT_MEAN=("AMT_CREDIT_LIMIT_ACTUAL", "mean"),
        CC_AMT_CREDIT_LIMIT_MAX=("AMT_CREDIT_LIMIT_ACTUAL", "max"),
        CC_AMT_DRAWINGS_ATM_MEAN=("AMT_DRAWINGS_ATM_CURRENT", "mean"),
        CC_AMT_DRAWINGS_CURRENT_MEAN=("AMT_DRAWINGS_CURRENT", "mean"),
        CC_AMT_PAYMENT_CURRENT_MEAN=("AMT_PAYMENT_CURRENT", "mean"),
        CC_SK_DPD_MEAN=("SK_DPD", "mean"),
        CC_SK_DPD_MAX=("SK_DPD", "max"),
        CC_SK_DPD_DEF_MEAN=("SK_DPD_DEF", "mean"),
        CC_SK_DPD_DEF_MAX=("SK_DPD_DEF", "max"),
    ).reset_index()
    
    # Credit utilization ratio
    cc_agg["CC_UTILIZATION_RATIO"] = (
        cc_agg["CC_AMT_BALANCE_MEAN"] / cc_agg["CC_AMT_CREDIT_LIMIT_MEAN"].replace(0, np.nan)
    ).astype(np.float32)
    
    # Count of unique credit card contracts
    cc_agg["CC_CONTRACT_COUNT"] = cc.groupby("SK_ID_CURR")["SK_ID_PREV"].nunique().values
    
    # Convert float columns
    float_cols = cc_agg.select_dtypes(include=["float64"]).columns
    cc_agg[float_cols] = cc_agg[float_cols].astype(np.float32)
    
    print(f"  Credit card features:    {cc_agg.shape[1] - 1:>3} features for {cc_agg.shape[0]:>7,} clients")
    return cc_agg


print("Aggregating credit card balance features...\n")
cc_features = aggregate_credit_card_features(cc)

del cc
gc.collect()
print(f"\n  Memory freed after credit_card_balance processing.")

## 6. Installments Payments Features

Aggregate installment payment history to the `SK_ID_CURR` level.

In [ ]:
def aggregate_installments_features(inst):
    """Aggregate installments_payments table to SK_ID_CURR level."""
    
    # --- Derived columns ---
    inst = inst.copy()
    
    # Payment difference: positive = overpayment, negative = underpayment
    inst["PAYMENT_DIFF"] = inst["AMT_PAYMENT"] - inst["AMT_INSTALMENT"]
    
    # Days before/after due: negative = early, positive = late
    inst["DAYS_LATE"] = inst["DAYS_ENTRY_PAYMENT"] - inst["DAYS_INSTALMENT"]
    
    # Late payment flag
    inst["LATE_PAYMENT_FLAG"] = (inst["DAYS_LATE"] > 0).astype(np.int8)
    
    # Payment ratio
    inst["PAYMENT_RATIO"] = (
        inst["AMT_PAYMENT"] / inst["AMT_INSTALMENT"].replace(0, np.nan)
    )
    
    inst_agg = inst.groupby("SK_ID_CURR").agg(
        INST_COUNT=("SK_ID_PREV", "count"),
        INST_PAYMENT_DIFF_MEAN=("PAYMENT_DIFF", "mean"),
        INST_PAYMENT_DIFF_MIN=("PAYMENT_DIFF", "min"),
        INST_PAYMENT_DIFF_MAX=("PAYMENT_DIFF", "max"),
        INST_PAYMENT_DIFF_SUM=("PAYMENT_DIFF", "sum"),
        INST_DAYS_LATE_MEAN=("DAYS_LATE", "mean"),
        INST_DAYS_LATE_MAX=("DAYS_LATE", "max"),
        INST_LATE_PAYMENT_COUNT=("LATE_PAYMENT_FLAG", "sum"),
        INST_LATE_PAYMENT_TOTAL=("LATE_PAYMENT_FLAG", "count"),
        INST_AMT_PAYMENT_MEAN=("AMT_PAYMENT", "mean"),
        INST_AMT_INSTALMENT_MEAN=("AMT_INSTALMENT", "mean"),
        INST_PAYMENT_RATIO_MEAN=("PAYMENT_RATIO", "mean"),
        INST_PAYMENT_RATIO_MIN=("PAYMENT_RATIO", "min"),
        INST_NUM_CONTRACTS=("SK_ID_PREV", "nunique"),
    ).reset_index()
    
    # Percentage of late payments
    inst_agg["INST_LATE_PAYMENT_RATE"] = (
        inst_agg["INST_LATE_PAYMENT_COUNT"] / inst_agg["INST_LATE_PAYMENT_TOTAL"]
    ).astype(np.float32)
    
    # Drop helper column
    inst_agg.drop(columns=["INST_LATE_PAYMENT_TOTAL"], inplace=True)
    
    # Convert float columns
    float_cols = inst_agg.select_dtypes(include=["float64"]).columns
    inst_agg[float_cols] = inst_agg[float_cols].astype(np.float32)
    
    print(f"  Installments features:   {inst_agg.shape[1] - 1:>3} features for {inst_agg.shape[0]:>7,} clients")
    return inst_agg


print("Aggregating installments payments features...\n")
inst_features = aggregate_installments_features(inst)

del inst
gc.collect()
print(f"\n  Memory freed after installments_payments processing.")

## 7. Merge All Features

Left join all aggregated feature tables onto the application tables using `SK_ID_CURR`. Clients without records in auxiliary tables will have NaN values for those features.

In [ ]:
def merge_all_features(app, bureau_feat, bb_feat, prev_feat, pos_feat, cc_feat, inst_feat, label=""):
    """Left join all feature tables onto application table."""
    
    df = app.copy()
    n_start = df.shape[1]
    
    feature_tables = [
        ("Bureau",          bureau_feat),
        ("Bureau Balance",  bb_feat),
        ("Previous App",    prev_feat),
        ("POS Cash",        pos_feat),
        ("Credit Card",     cc_feat),
        ("Installments",    inst_feat),
    ]
    
    for name, feat_df in feature_tables:
        cols_before = df.shape[1]
        df = df.merge(feat_df, on="SK_ID_CURR", how="left")
        cols_added = df.shape[1] - cols_before
        coverage = df[feat_df.columns[1]].notna().mean() * 100
        print(f"  + {name:20s} → {cols_added:>3} features added  (coverage: {coverage:>5.1f}%)")
    
    # Replace infinities with NaN
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    n_end = df.shape[1]
    print(f"\n  {label} final shape: {df.shape[0]:>10,} rows × {n_end:>3} cols  (+{n_end - n_start} features)")
    
    return df


print("Merging features — TRAIN set:\n")
train_features = merge_all_features(
    app_train, bureau_features, bb_features,
    prev_features, pos_features, cc_features, inst_features,
    label="Train"
)

print("\n" + "=" * 70)
print("\nMerging features — TEST set:\n")
test_features = merge_all_features(
    app_test, bureau_features, bb_features,
    prev_features, pos_features, cc_features, inst_features,
    label="Test"
)

# Clean up feature tables
del bureau, bureau_features, bb_features, prev_features, pos_features, cc_features, inst_features
gc.collect()
print("\nMemory freed after merge.")

In [ ]:
# Feature count summary
print("Feature Count Summary")
print("=" * 50)

# Count features by prefix
prefixes = {
    "Application (original)": lambda c: not any(c.startswith(p) for p in ["BUREAU_", "BB_", "PREV_", "POS_", "CC_", "INST_"]),
    "Bureau (BUREAU_)":       lambda c: c.startswith("BUREAU_"),
    "Bureau Balance (BB_)":   lambda c: c.startswith("BB_"),
    "Previous App (PREV_)":   lambda c: c.startswith("PREV_"),
    "POS Cash (POS_)":        lambda c: c.startswith("POS_"),
    "Credit Card (CC_)":      lambda c: c.startswith("CC_"),
    "Installments (INST_)":   lambda c: c.startswith("INST_"),
}

total = 0
for group, condition in prefixes.items():
    count = sum(1 for c in train_features.columns if condition(c))
    total += count
    print(f"  {group:30s} {count:>4} features")

print(f"  {'':30s} {'----':>4}")
print(f"  {'TOTAL':30s} {total:>4} features")

# Missing value overview
print(f"\nMissing Values Overview (train):")
missing_pct = (train_features.isnull().sum() / len(train_features) * 100)
print(f"  Columns with no missing:  {(missing_pct == 0).sum():>4}")
print(f"  Columns with <10% missing: {((missing_pct > 0) & (missing_pct < 10)).sum():>3}")
print(f"  Columns with >50% missing: {(missing_pct > 50).sum():>3}")
print(f"  Columns with >90% missing: {(missing_pct > 90).sum():>3}")

## 8. Save to Gold Layer

Save the final merged feature datasets to the gold layer as Parquet files.

In [ ]:
# Save train features
train_path = GOLD_DIR / "train_features.parquet"
train_features.to_parquet(train_path, engine="pyarrow", index=False)

# Save test features
test_path = GOLD_DIR / "test_features.parquet"
test_features.to_parquet(test_path, engine="pyarrow", index=False)

print("Gold layer saved:\n")
print(f"  {'File':40s} {'Rows':>10s}   {'Cols':>4s}   {'Size':>8s}")
print(f"  {'-'*40} {'-'*10}   {'-'*4}   {'-'*8}")

for name, path, df in [("train_features.parquet", train_path, train_features),
                        ("test_features.parquet", test_path, test_features)]:
    size_mb = path.stat().st_size / 1e6
    print(f"  {name:40s} {df.shape[0]:>10,}   {df.shape[1]:>4}   {size_mb:>7.1f}M")

# Verify saved files
print("\nVerification:")
train_check = pd.read_parquet(train_path)
test_check = pd.read_parquet(test_path)
print(f"  Train reload: {train_check.shape[0]:,} rows × {train_check.shape[1]} cols — {'OK' if train_check.shape == train_features.shape else 'MISMATCH'}")
print(f"  Test reload:  {test_check.shape[0]:,} rows × {test_check.shape[1]} cols — {'OK' if test_check.shape == test_features.shape else 'MISMATCH'}")
print(f"  TARGET in train: {'YES' if 'TARGET' in train_check.columns else 'NO'}")
print(f"  TARGET in test:  {'YES' if 'TARGET' in test_check.columns else 'NO  (expected)'}")

del train_check, test_check
gc.collect()
print("\nGold layer complete!")

## Summary

**What we did:**
1. Loaded all 8 silver layer tables from `data/processed/`
2. Engineered aggregated features from each auxiliary table:
   - **Bureau**: Credit history stats, active/closed counts, overdue proportions
   - **Bureau Balance**: DPD status counts and ratios per client
   - **Previous Application**: Application stats, approval rates, credit ratios
   - **POS Cash Balance**: DPD stats, completed contract counts
   - **Credit Card Balance**: Utilization ratios, drawing/payment patterns, DPD stats
   - **Installments Payments**: Payment differences, late payment rates
3. Merged all features onto application tables via left join on `SK_ID_CURR`
4. Handled infinities (replaced with NaN)
5. Saved gold layer datasets to `data/features/`

**Output files:**
- `data/features/train_features.parquet` — full training feature matrix with TARGET
- `data/features/test_features.parquet` — test feature matrix (no TARGET)

**Next:** NB04 — Feature Store Setup with Feast